# Cellpose segmentation

We will use the [Cellpose](https://www.cellpose.org/) model to segment the images. Cellpose is a generalist algorithm for cellular segmentation that can be applied to a wide range of cell types and imaging modalities.

In [ ]:
# Needs to be installed before running the notebook
!pip install git+https://github.com/FLClab/TiffWrapper.git

In [7]:
import random
import cellpose
import numpy
import pandas
import matplotlib

from cellpose import models, io, utils
from matplotlib import pyplot
from bioio import BioImage
import bioio_bioformats
from tiffwrapper import make_composite

# Create the Cellpose model instance with GPU enabled or disabled
model = models.CellposeModel(gpu=False)

Some utility functions are provided to help with the segmentation process, including functions to shuffle the masks for better visualization and to display the original image alongside the segmentation results.

In [ ]:
def shuffle_masks(masks):
    """
    Shuffle the labels in the masks array to randomize the label values.

    Parameters:
    -----------
    masks : numpy.ndarray
        The input masks array with labeled regions.

    Returns:
    --------
    numpy.ndarray
        The shuffled masks array with randomized label values.
    """
    unique_labels = numpy.unique(masks)[1:] # Exclude the background label (0)
    shuffled_labels = numpy.random.permutation(unique_labels)
    label_mapping = {old: new for old, new in zip(unique_labels, shuffled_labels)}
    
    shuffled_masks = numpy.copy(masks)
    for old_label, new_label in label_mapping.items():
        shuffled_masks[masks == old_label] = new_label
    
    return shuffled_masks

Here are the general steps that should be followed in order to segment the images using Cellpose:
1. Load the images using BioImage. Make sure to use the `reader=bioio_bioformats.Reader` option to read the images in the correct format.
1. Analyze the images using Cellpose to obtain the segmentation masks.

    - Before running the segmentation, resize the images to a smaller size (e.g., 256x256) to speed up the segmentation process. You can use the `skimage.transform.resize` function for this purpose. Or, you can use the `cellpose.transforms.resize_image` function provided by Cellpose.
    - After resizing, run the Cellpose model on the images to obtain the segmentation masks. The Cellpose model is already available with the `model` variable. You can use the `model.eval` method to run the model on the images and obtain the segmentation masks.
    - Plot the original image alongside the segmentation masks. This will allow you to ensure that the segmentation is working correctly and that the masks are aligned with the original image.

1. Extract the features from the segmentation masks using `skimage.measure.regionprops`. This will allow us to obtain properties such as the mean intensity of each cell in different channels. Extract the mean intensity of each cell in the CaMKII and EGFP channels, and store these values in a pandas DataFrame.

    - Plot the distribution of features (mean intensity values) for each channel. This will allow you to visualize the distribution of the features and identify any potential outliers or patterns in the data.
    - Plot the relationship between the mean intensity values in the CaMKII and EGFP channels. This will allow you to visualize the correlation between the two channels and identify any potential relationships between them.

1. Use KMeans clustering to group the cells based on their mean intensity values in the CaMKII and EGFP channels. Determine the optimal number of clusters using the silhouette score.
1. Based on the clustering results, create a new mask where each region is colored according to its cluster label. This will allow you to visualize the clustering results and identify any potential patterns or relationships between the clusters.